# From characters to attention: train a small language model

Browser Python lab: run in order; cells share variables. Charts come from the code you execute.


## 1 · Build a vocabulary and tensors

Character splitting is a teaching tokenizer, not BPE. Edit text and inspect vocabulary size and sequence length. Later cells reuse these variables.

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

import math, random
random.seed(7)
text = "hello model. hello world. hello model. "
vocab = sorted(set(text))
to_id = {c: i for i, c in enumerate(vocab)}
ids = [to_id[c] for c in text]
V, D = len(vocab), 4
embedding = [[random.uniform(-1,1) for _ in range(D)] for _ in vocab]
x = [embedding[i] for i in ids[:8]]
print("vocab:", to_id)
print("input IDs:", ids[:8])
print("embedding shape:", (len(x), D))
print("first vector:", x[0])

## 2 · Causal attention

For inspection Q, K and V all use embeddings; real layers use separate learned projections. The first row can only read itself.

In [ ]:
def softmax(z):
    e = [math.exp(v-max(z)) for v in z]
    return [v/sum(e) for v in e]
A, hidden = [], []
for i, q in enumerate(x):
    scores = [sum(a*b for a,b in zip(q,k))/D**0.5 for k in x[:i+1]]
    a = softmax(scores) + [0.0]*(len(x)-i-1)
    A.append(a)
    hidden.append([sum(a[j]*x[j][d] for j in range(len(x))) for d in range(D)])
print("attention shape:", (len(A),len(A)))
print("first row:", A[0])
print("hidden shape:", (len(hidden), D))
assert A[0] == [1.0]+[0.0]*(len(x)-1)
display_plot(list(range(len(A))), A[-1], "Last query attention", "key position", "weight")

## 3 · Real gradient training

Train a separate bigram model that predicts the next character from the current one. It does not use the previous attention cell and is not a Transformer; the complete cross-entropy gradient is visible.

In [ ]:
W = [[0.0]*V for _ in range(V)]
pairs = list(zip(ids[:-1], ids[1:]))
learning_rate, epochs = 4.0, 120
losses = []
for epoch in range(epochs):
    grad = [[0.0]*V for _ in range(V)]
    loss = 0.0
    for source, target in pairs:
        p = softmax(W[source])
        loss -= math.log(max(p[target],1e-12))/len(pairs)
        for j in range(V):
            grad[source][j] += (p[j]-(j==target))/len(pairs)
    for i in range(V):
        for j in range(V):
            W[i][j] -= learning_rate*grad[i][j]
    losses.append(loss)
print("parameters:", V*V)
print("initial / final training NLL:", losses[0], losses[-1])
print("This measures training fit, not held-out generalization.")
display_plot(list(range(epochs)), losses, "Training cross-entropy", "epoch", "NLL")

## 4 · Inputs, outputs and sampling

Change temperature and compare probabilities. The model learns neighboring-character statistics, not world knowledge. The sampling seed is fixed for comparison.

In [ ]:
temperature = 0.7
assert temperature > 0
random.seed(11)
current = to_id['h']
generated = [vocab[current]]
print("logits shape:", (1, V))
p = softmax([v/temperature for v in W[current]])
print("top next characters:", sorted(zip(vocab,p),key=lambda z:-z[1])[:5])
for _ in range(60):
    p = softmax([v/temperature for v in W[current]])
    current = random.choices(range(V), weights=p)[0]
    generated.append(vocab[current])
print("".join(generated))

## 5 · Optional: load real MiniLM inside this notebook

Running this cell separately downloads about 23 MB of weights plus runtime from Hugging Face and jsDelivr. Inference runs locally. This is an English embedding model, separate from the bigram trained above. Run all excludes this cell; it requires the blog-provided browser_models bridge. Edit the two sentences and inspect real tokens, masks and 384-dimensional outputs.

In [ ]:
import json
from browser_models import embed
texts = ["A cat is sleeping on the sofa.", "A kitten is resting on a couch."]
result = json.loads(await embed(texts))
print("actual input IDs:", result['ids'])
print("actual tokens:", result['tokens'])
print("attention masks:", result['masks'])
print("output shape:", result['dims'])
print("cosine similarity:", result['cosine'])
vector = result['embeddings'][0]
display_plot(list(range(32)), vector[:32], "Real MiniLM embedding (first 32 dimensions)", "dimension", "value")
